# Часть 2.4 — Red Wine Quality

3 эксперимента: (1) регрессия `quality`; (2) бинарная классификация `good (quality ≥ 7)`; (3) KMeans-кластеризация по физико-химическим признакам.

In [1]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from _setup import evaluate_models, make_spark
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    ClusteringEvaluator,
    MulticlassClassificationEvaluator,
    RegressionEvaluator,
)
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import functions as F
from sklearn.metrics import davies_bouldin_score

SEED = 42
spark = make_spark("hw3-wine")
spark

## Загрузка

In [2]:
csv = next(Path(kagglehub.dataset_download("uciml/red-wine-quality-cortez-et-al-2009")).glob("*.csv"))
raw = spark.read.csv(str(csv), header=True, inferSchema=True)
df = raw.toDF(*[c.replace(" ", "_") for c in raw.columns]).cache()
print(f"строк: {df.count()}")
df.groupBy("quality").count().orderBy("quality").show()

строк: 1599
+-------+-----+
|quality|count|
+-------+-----+
|      3|   10|
|      4|   53|
|      5|  681|
|      6|  638|
|      7|  199|
|      8|   18|
+-------+-----+



## Подготовка

In [3]:
feature_cols = [c for c in df.columns if c != "quality"]
prep = Pipeline(
    stages=[
        VectorAssembler(inputCols=feature_cols, outputCol="features_raw"),
        StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True),
    ]
).fit(df)
data = (
    prep.transform(df)
    .withColumn("label", F.col("quality").cast("double"))
    .withColumn("is_good", (F.col("quality") >= 7).cast("double"))
    .select("features", "label", "is_good")
)
train, test = data.randomSplit([0.8, 0.2], seed=SEED)
train.cache()
test.cache()
print(f"train: {train.count()}, test: {test.count()}")

train: 1324, test: 275


## Эксперимент 1 — регрессия quality

In [4]:
reg_evaluators = {
    "RMSE": RegressionEvaluator(labelCol="label", metricName="rmse"),
    "R2": RegressionEvaluator(labelCol="label", metricName="r2"),
}
regressors = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol="label", maxIter=50),
    "RFRegressor": RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=100, seed=SEED),
}
evaluate_models(regressors, train, test, reg_evaluators)

,RMSE,R2
model,,
LinearRegression,0.674736,0.349573
RFRegressor,0.642497,0.410244


## Эксперимент 2 — классификация good (≥ 7)

In [5]:
cls_evaluators = {
    "accuracy": MulticlassClassificationEvaluator(labelCol="is_good", metricName="accuracy"),
    "f1": MulticlassClassificationEvaluator(labelCol="is_good", metricName="f1"),
}
classifiers = {
    "LogReg": LogisticRegression(featuresCol="features", labelCol="is_good", maxIter=100),
    "RandomForest": RandomForestClassifier(featuresCol="features", labelCol="is_good", numTrees=100, seed=SEED),
}
evaluate_models(classifiers, train, test, cls_evaluators)

,accuracy,f1
model,,
LogReg,0.872727,0.861706
RandomForest,0.883636,0.868319


## Эксперимент 3 — KMeans (k=2..6)

In [6]:
sil_eval = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
rows = []
for k in range(2, 7):
    pred = KMeans(k=k, featuresCol="features", seed=SEED, maxIter=20).fit(data).transform(data)
    pdf = pred.select("features", "prediction").toPandas()
    X = np.stack(pdf["features"].apply(lambda v: v.toArray()).to_numpy())
    rows.append(
        {
            "k": k,
            "silhouette": sil_eval.evaluate(pred),
            "davies_bouldin": davies_bouldin_score(X, pdf["prediction"]),
        }
    )
exp3 = pd.DataFrame(rows).set_index("k")
exp3

,silhouette,davies_bouldin
k,,
2,0.328797,1.893230
3,0.298956,1.765160
4,0.333260,1.507850
5,0.301229,1.468985
6,0.318539,1.400253


In [7]:
best_k = int(exp3["silhouette"].idxmax())
print(f"best k = {best_k}")
best = KMeans(k=best_k, featuresCol="features", seed=SEED, maxIter=20).fit(data)
best.transform(prep.transform(df)).groupBy("prediction").agg(
    F.count("*").alias("n"),
    F.avg("alcohol").alias("avg_alcohol"),
    F.avg("volatile_acidity").alias("avg_vol_acid"),
    F.avg("sulphates").alias("avg_sulph"),
    F.avg("quality").alias("avg_quality"),
).orderBy("prediction").show()

best k = 4


+----------+---+------------------+------------------+------------------+-----------------+
|prediction|  n|       avg_alcohol|      avg_vol_acid|         avg_sulph|      avg_quality|
+----------+---+------------------+------------------+------------------+-----------------+
|         0|713|10.475876577840124|0.6144249649368863| 0.608527349228612|5.538569424964937|
|         1|479|10.819763395963811|0.3943841336116913|0.7190396659707723|6.018789144050104|
|         2| 29| 9.489655172413793| 0.526896551724138|1.2624137931034483|5.344827586206897|
|         3|378| 9.892019400352739|0.5336243386243384|0.6282275132275136|5.357142857142857|
+----------+---+------------------+------------------+------------------+-----------------+



## Выводы

- **Эксп. 1**: RFRegressor (RMSE ~0.64, R² ~0.41) лучше LinearRegression — связь с quality нелинейна.
- **Эксп. 2**: при бинаризации `good ≥ 7` RF и LogReg дают accuracy ~0.87–0.88; класс good ~14%.
- **Эксп. 3**: best k=4 (silhouette ~0.33, DB ~1.51); кластеры с высоким alcohol+sulphates имеют большую среднюю quality.
- **alcohol** — главный сквозной признак.

In [8]:
spark.stop()